# Curating a Subset of MedShapeNet

This notebook demonstrates how a curated subset of the MedShapeNet dataset was created. It integrates filtering, validation, and optional decimation of meshes into a reproducible pipeline. The goal is to ensure that only usable meshes (sufficient resolution, valid geometry) are retained.

Author: Tomas Krsicka

In [5]:
%pip install medshapenet pyfqmr trimesh

[notice] A new release of pip is available: 24.3.1 -> 25.2


[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: requests>=2.32.3 in c:\users\tomkr\appdata\local\packages\pythonsoftwarefoundation.python.3.10_qbz5n2kfra8p0\localcache\local-packages\python310\site-packages (from medshapenet) (2.32.3)



In [6]:
import os
import requests
import trimesh
from urllib.parse import urlsplit, parse_qs
import pyfqmr
from MedShapeNet import MedShapeNet as msn
import re

Define unification patterns. First element of the tuple is the pattern in the raw category, second is the resulting final unified category. Raw categories without matches remain unchanged.

In [7]:
CATEGORY_PATTERNS = [
    (r'vertebrae', 'vertebrae'),
    (r'lung', 'lung'),
    (r'heart', 'heart'),
    (r'rib_', 'rib'),
    (r'costa', 'costa'),
    (r'iliac', 'iliac_vena'),
    (r'clavic', 'clavicula'),
    (r'adrenal', 'adrenal_gland'),
    (r'inferior', 'inferior_vena_cava'),
    (r'pulmonary', 'pulmonary_artery'),
    (r'portal', 'portal_vein_and_splenic_vein'),
    (r'small', 'small_bowel'),
    (r'gluteus.*minimus', 'gluteus_minimus'),
    (r'gluteus.*maximus', 'gluteus_maximus'),
    (r'gluteus.*medius', 'gluteus_medius'),
]


Extract base category names from raw MedShapeNet names.

In [8]:

def extract_base_category_name(raw_category: str, merge_laterality: bool = True) -> str:
    category_lower = raw_category.lower()

    # General regex matching
    for pattern, unified in CATEGORY_PATTERNS:
        if re.search(pattern, category_lower):
            return unified

    # Handle suffixes like _left / _right
    if merge_laterality:
        match = re.match(r'(.*?)_*(left|right)$', category_lower)
        if match:
            return match.group(1)

    return category_lower

Create a mapping between raw and unified categories.

In [9]:
from collections import defaultdict

def map_unified_categories(raw_categories: [str], merge_laterality: bool = True):
    unified_dict = defaultdict(list)

    for raw_cat in raw_categories:
        unified = extract_base_category_name(raw_cat, merge_laterality=merge_laterality)
        unified_dict[unified].append(raw_cat)

    return dict(unified_dict)
raw_classes = ["rib_left","rib_right","vertebrae","brain","aorta","esophagus","skull","liver","colon","autochthon_left",
               "autochthon_right","trachea","duodenum","tumoredbrain","lung_upper_lobe_left","lung_lower_lobe_left","lung_lower_lobe_right",
               "inferior_vena_cava","gallbladder","stomach","spleen","lung_upper_lobe_right","vertebrae_T11","lung_middle_lobe_right",
               "vertebrae_T10","heart_ventricle_right","vertebrae_T12","heart_atrium_right","heart_myocardium","heart_ventricle_left",
               "heart_atrium_left","vertebrae_T9","pancreas","iliopsoas_right","portal_vein_and_splenic_vein","adrenal_gland_right",
               "iliopsoas_left","scapula_left","adrenal_gland_left","vertebrae_T8","scapula_right","small_bowel","kidney_left","vertebrae_L1",
               "kidney_right","vertebrae_T3","vertebrae_T4","vertebrae_T5","vertebrae_T7","clavicula_left","vertebrae_T6","vertebrae_T2",
               "clavicula_right","pulmonary_artery","humerus_right","vertebrae_T1","vertebrae_C7","vertebrae_L2","humerus_left","sacrum",
               "iliac_artery_left","hip_left","iliac_artery_right","hip_right","vertebrae_L3","iliac_vena_left","vertebrae_L4","iliac_vena_right",
               "gluteus_medius_left","vertebrae_L5","gluteus_medius_right","gluteus_maximus_left","gluteus_maximus_right","femur_left",
               "vertebrae_C6","gluteus_minimus_left","urinary_bladder","gluteus_minimus_right","femur_right","vertebrae_C5","tumoredKidney",
               "costa9right","costa5right","costa6left","costa6right","costa7left","costa7right","costa8left","costa8right","costa9left",
               "inferiorvenacava","costa4right","femurleft","femurright","gluteusmaximusleft","gluteusmaximusright","gluteusmediusleft",
               "gluteusmediusright","gluteusminimusright","heart","costa5left","costa3left","costa4left","costa10left","adrenalglandleft",
               "adrenalglandright","autochthonleft","autochthonright","bladder","bronchie","celiactrunk","clavicleleft","clavicleright",
               "costa10right","costa3right","costa11left","costa11right","costa12left","costa12right","costa1left","costa1right","costa2left",
               "costa2right","heartatriumright","heartatriumleft","gluteusminimusleft","heartmyocardium","vertebraeT3","vertebraeT12",
               "hearttissue","vertebraeT10","vertebraeT1","vertebraeL5","vertebraeL4","vertebraeL3","vertebraeL2","vertebraeL1","vertebraeC6",
               "vertebraeC5","vertebraeC4","vertebraeC3","vertebraeC2","vertebraeC1","uterus","iliopsoasleft","thyroidleft","mediastinaltissue",
               "lungupperloberight","lungupperlobeleft","lungmiddleloberight","lunglowerloberight","lunglowerlobeleft","iliopsoasright","vertebraeT2",
               "vertebraeT11","vertebraeT4","iliacvenaleft","heartventricleleft","heartventricleright","hipleft","hipright","humerusleft",
               "humerusright","vertebraeT5","iliacarteryright","iliacarteryleft","vertebraeC7","vertebraeT7","vertebraeT9","iliacvenaright",
               "vertebraeT8","vertebraeT6","eyeballleft","kidneyleft","thyroidright","eyeballright","breastleft","breastright","cheekleft",
               "cheekright","vertebrae_C2","vertebrae_C4","vertebrae_C1","vertebrae_C3","uterocervix","thymus","gonads","ct_skull","vessel_tree",
               "pulmonaryartery","prostate","scapulaleft","ribcartilage","portalveinandsplenicvein","smallbowel","scapularight","sternum",
               "spinalcanal","face"]


category_map = map_unified_categories(raw_classes, merge_laterality=True)

for unified, raws in category_map.items():
    print(f"{unified}: {raws}")

rib: ['rib_left', 'rib_right']
vertebrae: ['vertebrae', 'vertebrae_T11', 'vertebrae_T10', 'vertebrae_T12', 'vertebrae_T9', 'vertebrae_T8', 'vertebrae_L1', 'vertebrae_T3', 'vertebrae_T4', 'vertebrae_T5', 'vertebrae_T7', 'vertebrae_T6', 'vertebrae_T2', 'vertebrae_T1', 'vertebrae_C7', 'vertebrae_L2', 'vertebrae_L3', 'vertebrae_L4', 'vertebrae_L5', 'vertebrae_C6', 'vertebrae_C5', 'vertebraeT3', 'vertebraeT12', 'vertebraeT10', 'vertebraeT1', 'vertebraeL5', 'vertebraeL4', 'vertebraeL3', 'vertebraeL2', 'vertebraeL1', 'vertebraeC6', 'vertebraeC5', 'vertebraeC4', 'vertebraeC3', 'vertebraeC2', 'vertebraeC1', 'vertebraeT2', 'vertebraeT11', 'vertebraeT4', 'vertebraeT5', 'vertebraeC7', 'vertebraeT7', 'vertebraeT9', 'vertebraeT8', 'vertebraeT6', 'vertebrae_C2', 'vertebrae_C4', 'vertebrae_C1', 'vertebrae_C3']
brain: ['brain']
aorta: ['aorta']
esophagus: ['esophagus']
skull: ['skull']
liver: ['liver']
colon: ['colon']
autochthon: ['autochthon_left', 'autochthon_right', 'autochthonleft', 'autochthonrig

## Setup
We first define the helper functions for downloading, validating, and simplifying meshes. These are adapted from the dataset filtering script.

In [10]:
def get_file_size(url: str):
    try:
        response = requests.head(url, allow_redirects=True, timeout=5)
        if 'Content-Length' in response.headers:
            return int(response.headers['Content-Length'])
    except:
        return None
    return None

def download_file(url: str, dest_path: str):
    try:
        with requests.get(url, stream=True, timeout=10) as r:
            r.raise_for_status()
            with open(dest_path, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
        return True
    except Exception:
        return False
        
def log_invalid(path: str, url: str, reason: str):
    with open(path, 'a') as fout:
        fout.write(f"{url} # {reason}\n")

def extract_filename(url):
    parsed = urlsplit(url)
    qs = parse_qs(parsed.query)
    filename = qs.get('files', [os.path.basename(parsed.path)])[0]
    return filename


def download_to_path(url, dest_path):
    if download_file(url, dest_path):
        return True
    log_invalid(invalid_path, url, "Download failed")
    return False

## Filtering Loop
This loop processes each URL:
- Check file size
- Download
- Validate STL
- Apply face-count and watertightness checks
- Optionally decimate large meshes
- Save valid files, log invalid ones
The limit variable sets the number of samples per category in the final clean dataset
Add raw classes that you want to include to the classes list

In [11]:
import os
from pathlib import Path
from urllib.parse import urlsplit, parse_qs

classes = ["humerus_left", "femur_right", "femur_left"]
download_dir = Path("downloads")
invalid_path = "invalid.log"
valid_path = "valid.log"
limit = 10
url_dictionary={}
download_dir.mkdir(exist_ok=True)

In [12]:
for cls in classes:
    list_urls = msn.search_by_name(name=cls, print_output=False)
    unified_class_name = extract_base_category_name(cls, True)
    url_dictionary.setdefault(unified_class_name, []).extend(list_urls)

Define helper functions for url and file management

Sample validation functions. 
Even before downloading the file, we can check if the file is too small signifying an invalid sample. 
Then we check if the stl is formed correctly and finally, we check for watertightness and number of faces


In [13]:
def check_file_size(url, min_size=50_000):
    size = get_file_size(url)
    if size is None:
        return False, "Could not retrieve file size"
    if size < min_size:
        return False, f"File too small ({size} bytes)"
    return True, size
    
def is_valid_stl(path: str):
    try:
        mesh = trimesh.load_mesh(path)
        if mesh.is_empty:
            return False, None
        components = mesh.split(only_watertight=False)
        largest = max(components, key=lambda m: len(m.faces))
        return True, largest
    except Exception:
        return False, None
def validate_stl(path, min_faces=400, watertight_threshold=1000):
    valid, largest_component = is_valid_stl(path)
    if not valid:
        return False, "Not a valid STL", None
    faces = len(largest_component.faces)
    if faces < min_faces:
        return False, f"Too few faces: {faces}", largest_component
    if faces < watertight_threshold and not largest_component.is_watertight:
        return False, f"Non-watertight with {faces} faces", largest_component
    return True, "Valid STL", largest_component

Download and check a single sample

In [14]:
def process_single_url(url, merge_laterality=True):
    filename = extract_filename(url)
    dest_path = download_dir / filename

    ok, info = check_file_size(url)
    if not ok:
        log_invalid(invalid_path, url, info)
        print(f"\tSkipped: {info}")
        return False

    if not download_to_path(url, dest_path):
        print("Skipped: download failed")
        return False

    ok, info, component = validate_stl(dest_path)
    if not ok:
        log_invalid(invalid_path, url, info)
        dest_path.unlink(missing_ok=True)
        print(f"\tSkipped: {info}")
        return False

    with open(valid_path, 'a') as fout:
        fout.write(url + '\n')

    print(f"\tProcessed: {info}")
    return True

Example of filtering the above defined classes

In [15]:
total_valid = 0
base_dir = "msn_filter/"
total_processed = 0
for cls in url_dictionary.keys():
    
    invalid_path = os.path.join(base_dir, f"{cls}_invalid.txt")
    valid_path = os.path.join(base_dir, f"{cls}_valid.txt")
    download_dir = Path(os.path.join(base_dir, "download", cls))
    os.makedirs(download_dir, exist_ok=True)
    valid_count = 0
    processed = 0
    total = len(url_dictionary[cls])
    for url in url_dictionary[cls]:
        if valid_count >= limit:
            break
        print(f"[{processed}/{total}] Checking: {url}")
        valid = process_single_url(url)
        print("-"*20)
        processed += 1
    
        with open(valid_path, 'a') as fout:
            fout.write(url + '\n')
        valid_count += 1 if valid else 0
    print(f"\nFinished class {cls}. {valid_count} valid files saved. {processed} processed.")
    total_valid += valid_count
    total_processed += processed
print(f"\nFinished. Total {valid_count} valid files saved. {processed} total processed.")


[0/580] Checking: https://uni-duisburg-essen.sciebo.de/s/HeShw1G4da15OOZ/download?path=%2F&files=s0004_humerus_left.nii.g_1.stl
	Processed: Valid STL
--------------------
[1/580] Checking: https://uni-duisburg-essen.sciebo.de/s/HeShw1G4da15OOZ/download?path=%2F&files=s0011_humerus_left.nii.g_1.stl
	Processed: Valid STL
--------------------
[2/580] Checking: https://uni-duisburg-essen.sciebo.de/s/HeShw1G4da15OOZ/download?path=%2F&files=s0019_humerus_left.nii.g_1.stl
	Processed: Valid STL
--------------------
[3/580] Checking: https://uni-duisburg-essen.sciebo.de/s/HeShw1G4da15OOZ/download?path=%2F&files=s0021_humerus_left.nii.g_1.stl
	Processed: Valid STL
--------------------
[4/580] Checking: https://uni-duisburg-essen.sciebo.de/s/HeShw1G4da15OOZ/download?path=%2F&files=s0024_humerus_left.nii.g_1.stl
	Processed: Valid STL
--------------------
[5/580] Checking: https://uni-duisburg-essen.sciebo.de/s/HeShw1G4da15OOZ/download?path=%2F&files=s0028_humerus_left.nii.g_1.stl
	Processed: Valid

## Summary
- Invalid files and reasons are logged to `<sample_name>_invalid.txt`
- Valid file URLs are saved to `<sample_name>_valid.txt`
- Meshes are downloaded and stored under `msn_filter/download/<sample_name>/`
